# OOD prep - OneStopEnglish (HF) to Drive

This notebook downloads `onestop_english` from Hugging Face, maps labels to `education_level`, and saves a CSV formatted for `ood_run_static_metrics.py`.


In [ ]:
%pip install -q datasets pandas


In [ ]:
import os

MY_DRIVE_SUBDIR = 'BeyondFK'
OOD_SUBDIR = 'ood'
OUTPUT_FILENAME = 'onestopenglish_prepared.csv'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = os.path.join('/content/drive/MyDrive', MY_DRIVE_SUBDIR)
except Exception:
    print('Not Colab - using current directory.')
    BASE_DIR = '.'

OOD_DIR = os.path.join(BASE_DIR, OOD_SUBDIR)
os.makedirs(OOD_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OOD_DIR, OUTPUT_FILENAME)

print('BASE_DIR  :', os.path.abspath(BASE_DIR))
print('OOD_DIR   :', os.path.abspath(OOD_DIR))
print('OUTPUT_CSV:', os.path.abspath(OUTPUT_CSV))


In [ ]:
from datasets import load_dataset
import pandas as pd

HF_DATASET = 'onestop_english'
HF_SPLIT = 'train'

ds = load_dataset(HF_DATASET, split=HF_SPLIT)
df_raw = ds.to_pandas()
print('Columns:', df_raw.columns.tolist())
print('Rows:', len(df_raw))
df_raw.head(3)


In [ ]:
def map_labels_onestop_int(series):
    m = {0: 'elementary', 1: 'middle', 2: 'high', '0': 'elementary', '1': 'middle', '2': 'high'}
    out = []
    for v in series:
        if pd.isna(v):
            out.append(None)
            continue
        if isinstance(v, str) and v.strip().isdigit():
            v = int(v.strip())
        out.append(m.get(v))
    return pd.Series(out, index=series.index)

def build_prepared_df(df, text_col='text', label_col='label'):
    out = df.copy()
    out['full_text'] = out[text_col].astype(str)
    out['education_level'] = map_labels_onestop_int(out[label_col])
    for c in ('text_question', 'text_solution', 'text_lecture'):
        if c not in out.columns:
            out[c] = ''
        else:
            out[c] = out[c].fillna('').astype(str)
    out = out.dropna(subset=['education_level'])
    out = out[out['full_text'].str.strip().astype(bool)]
    return out

df_prep = build_prepared_df(df_raw)
print('Prepared rows:', len(df_prep))
print(df_prep['education_level'].value_counts())


In [ ]:
df_prep.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)
print('Now copy this CSV to HPC and run ood_run_static_metrics.py')
